# NeuroVision-X — Kaggle training driver

Thin driver. No training logic lives here: this notebook attaches the code and the data,
composes the real Hydra config, and calls `scripts/train.py::run_training`. Everything else
is in `src/neurovision/`, tested on the Mac's CPU.

**Before running:** notebook settings → GPU accelerator ON, internet ON (both need a
phone-verified account). Attach the preprocessed dataset. For a long run use
*Save Version → Save & Run All (Commit)*, never the interactive session.

Every cell below cell 1 fails immediately and with a readable message if a path is wrong —
the point is to find out in the first minute, not 40 minutes in.

Full workflow, including how to upload the dataset and chain sessions: `docs/kaggle_workflow.md`.

## 1. Session config — the only cell you edit

In [ ]:
# Set BEFORE torch is imported anywhere (this is the first cell that runs).
# Cheap insurance against allocator fragmentation; no recompute cost.
import os

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Note the two different accounts: GitHub is AmishhYadav, Kaggle is amishyadav123.
REPO_URL   = "https://github.com/AmishhYadav/NeuroVision-X.git"

# PINNED TO A SHA, not "main". docs/experiments.md note 4 on the published
# baseline_unet3d row records this rule and why it exists: that run cloned
# "main", W&B captured no git metadata, and its exact source revision is now
# unrecoverable. A long run also spans hours during which main can move, so
# "main" lets a resumed second session clone different code than the first.
# REPLACED PER RUN -- see the checklist in the run block below.
GIT_REF    = "__PIN_SHA__"

DATA_SLUG  = "amishyadav123/neurovision-brats-prep"  # attached preprocessed dataset
CKPT_SLUG  = None  # previous notebook output / checkpoint dataset to resume from; None = fresh run

# --- RUN 1 of 3: baseline_unet3d ----------------------------------------
# The Milestone-1 baseline, and the number the fusion model has to be
# competitive with. Cheapest of the three, so it runs first: if anything about
# the 64^3 / 80-epoch schedule is wrong, it surfaces here for ~3 h rather than
# during a 24 h fusion run.
#
# Priced from measurement, not arithmetic: the published 200-epoch row ran at
# 0.082 h/epoch at 96^3, and the 64^3 patch is 0.296 of that volume, so
# ~0.024 h/epoch x 80 = ~2 h of training plus 8 validation passes. Comfortably
# inside one 12 h session, so no resume is needed and CKPT_SLUG stays None.
#
# Verified on CPU before pushing: composes to unet3d / dice_ce / no deep
# supervision / 64^3 / 80 epochs / val_interval 10 / overfit_n None; forward
# emits 3 channels (never 4 -- 4 is the class count AND the modality count, so
# it is the easy thing to get wrong); loss 1.465 and grad norm 0.522 at init;
# 0.16 GB of activations per patch, so ~1.6 GiB of the T4's 14.56 GiB.
#
# EXPERIMENT is None because the experiment file already sets experiment_name.
# Passing it as well is harmless here but wrong in general -- an explicit
# experiment_name= override beats a config group's own value, which would
# silently rename the run and redirect its checkpoints.
EXPERIMENT = None

# "online" needs WANDB_API_KEY attached to THIS notebook under Add-ons ->
# Secrets. A kernel created by `kaggle kernels push` has none, so this is
# "offline": the full run is written to /kaggle/working/wandb/ and lands in
# the notebook output for a later `wandb sync`, with every metric intact.
# Never "disabled" -- that would drop the W&B half of the metrics entirely.
WANDB_MODE = "offline"

OVERRIDES  = [
    "+experiment=baseline_unet3d",
    # Kaggle gives 4 vCPU; 2 workers matches what every previous run used.
    "data.num_workers=2",
]

# --- Remaining runs, in order -------------------------------------------
# Re-pin GIT_REF each time, and record the SHA in the run's
# docs/experiments.md row.
#
# 2. neurovision (~23.7 h -> TWO sessions; on the second set CKPT_SLUG to this
#    notebook version's output and confirm the log says "RESUME: ... epoch N"):
#      EXPERIMENT = None
#      OVERRIDES  = ["+experiment=neurovision", "data.num_workers=2"]
# 3. ablation_content_only_gate (~23.7 h -> two sessions). The load-bearing P2
#    run. Its overrides must match run 2 EXACTLY apart from the experiment
#    name, or the comparison stops isolating the ambiguity signal:
#      EXPERIMENT = None
#      OVERRIDES  = ["+experiment=ablation_content_only_gate", "data.num_workers=2"]


## 2. Code + dependencies

Clone rather than `pip install git+...` alone: `configs/` and `scripts/` are not package data,
and Hydra needs the config tree on disk. The clone is then installed editable with `--no-deps`,
so `import neurovision` works without `PYTHONPATH` — same as local dev.

`torch`/`torchvision` are stripped from `requirements.txt` on purpose: the Kaggle image ships a
CUDA-matched build, and installing the pinned wheel over it silently loses the GPU. The assert
catches that, and a GPU that was never enabled, in ~30 seconds.

If `pip install -e` ever fails on `requires-python` (Kaggle moving off 3.11), replace that line
with `import sys; sys.path.insert(0, "/kaggle/working/repo/src")`.

In [ ]:
!git clone -q --depth 1 -b {GIT_REF} {REPO_URL} /kaggle/working/repo
# Build the Kaggle install list from requirements.txt minus its own
# `# kaggle-exclude:` line -- single source of truth, no second pinned file to
# drift. Everything excluded is either already in the Kaggle image (and ABI-
# linked to the rest of it) or dev-only.
import pathlib
import re
import subprocess
import sys

_req = pathlib.Path("/kaggle/working/repo/requirements.txt").read_text()
_excl = {w for m in re.findall(r"^#\s*kaggle-exclude:\s*(.+)$", _req, re.M) for w in m.split()}
_keep = [
    ln for ln in _req.splitlines()
    if ln.strip() and not ln.lstrip().startswith("#")
    and re.split(r"[=<>~!\[]", ln.strip())[0].strip() not in _excl
]
print("installing:", " ".join(_keep))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_keep], check=True)

In [ ]:
import torch

# sys.path, NOT `pip install -e`. Kaggle runs Python 3.12 and pyproject.toml
# pins requires-python = ">=3.11,<3.12", so pip REFUSES the editable install
# ("Package 'neurovision-x' requires a different Python"). A `!pip` failure
# does not stop a notebook cell, so that error scrolled past and a previous run
# died four cells later on ModuleNotFoundError. sys.path needs no metadata
# check and works on any interpreter.
sys.path.insert(0, "/kaggle/working/repo/src")
sys.path.insert(0, "/kaggle/working/repo/scripts")
import neurovision  # noqa: F401  -- verify HERE, not four cells later
import scipy.ndimage  # noqa: F401 -- canary: breaks if our numpy pin overwrote Kaggle's

assert torch.cuda.is_available(), "No CUDA: GPU accelerator off, or pip replaced Kaggle's CUDA torch build."
_name = torch.cuda.get_device_name(0)
_cap = "sm_%d%d" % torch.cuda.get_device_capability(0)
# is_available() is NOT sufficient, and this is not hypothetical: on a Kaggle
# P100 it returns True while every kernel launch fails, because the stock torch
# build no longer targets sm_60 (min sm_70). Only executing something is honest.
try:
    (torch.randn(64, 64, device="cuda") @ torch.randn(64, 64, device="cuda")).sum().item()
except Exception as exc:
    raise RuntimeError(
        f"{_name} ({_cap}) reports CUDA available but cannot run a kernel: {exc}\n"
        "Set machine_shape=NvidiaTeslaT4 (sm_75); the P100 is sm_60."
    ) from exc

import monai
import numpy as np

print(f"{_name}  {_cap}")
print(f"torch {torch.__version__} | numpy {np.__version__} | monai {monai.__version__} "
      f"| python {sys.version.split()[0]}")

## 3. Environment — W&B

`WANDB_MODE` in cell 1 picks one of three:

- **`"online"`** — needs `WANDB_API_KEY` as a Kaggle Secret, added under Add-ons → Secrets **and
  attached to this notebook**. A secret on your account but not attached to this kernel fails
  with `No user secrets exist for kernel id <id> and label <label>`. The label is exact and
  case-sensitive; set `SECRET_LABEL` to whatever you actually named it.
- **`"offline"`** — no secret needed. The full run is written to `/kaggle/working/wandb/` and
  saved with the notebook output; `wandb sync <dir>` uploads it later with every metric intact.
- **`"disabled"`** — no logging at all.

Never paste the key into a cell: committed notebook versions are stored with their source.

In [ ]:
import os

# Exact, case-sensitive label of the Kaggle Secret. Must match what you named it
# in Add-ons -> Secrets AND be attached to THIS notebook.
SECRET_LABEL = "WANDB_API_KEY"

# Only "online" needs a Kaggle Secret. "offline" writes the complete run to
# /kaggle/working/wandb/ with no API key at all -- it lands in the notebook
# output, and `wandb sync <dir>` from your Mac uploads it afterwards with the
# curves intact. Use it when the secret is unavailable; it is NOT a downgrade
# in what gets recorded, only in when it appears in the dashboard.
if WANDB_MODE == "online":
    from kaggle_secrets import UserSecretsClient

    # Not wrapped in try/except: a missing secret on a real run means losing the
    # run's curves, so it must stop here, not 11 hours in.
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret(SECRET_LABEL)
    print(f"W&B online (secret label {SECRET_LABEL!r})")
elif WANDB_MODE == "offline":
    os.environ["WANDB_DIR"] = "/kaggle/working"
    OVERRIDES = [*OVERRIDES, "wandb.mode=offline"]
    print("W&B OFFLINE -- run written to /kaggle/working/wandb/. "
          "Upload later with: wandb sync <that dir>")
else:
    OVERRIDES = [*OVERRIDES, "wandb.mode=disabled"]
    print("W&B DISABLED -- nothing will be logged.")

## 4. Resolve the attached dataset

Layout is what `scripts/package_for_kaggle.py` builds: `preprocessed/<case>/{image,label}.npy`,
`metadata.csv`, `splits.yaml`. A wrong or unattached dataset raises here, listing what *is*
mounted so the fix is obvious.

In [ ]:
import shutil
from pathlib import Path

# Discovered, not assumed. A dataset does NOT reliably mount at
# /kaggle/input/<slug>: an earlier run of this notebook found the whole of
# /kaggle/input to be just ['datasets'], i.e. one level deeper than the
# documented layout. So look for the shape we need -- a directory holding both
# preprocessed/ and splits.yaml -- across the first few levels, rather than
# hardcoding a path Kaggle is free to change.
_roots = [Path("/kaggle/input")]
_hits = sorted(
    {
        p.parent
        for pat in ("splits.yaml", "*/splits.yaml", "*/*/splits.yaml", "*/*/*/splits.yaml")
        for r in _roots
        for p in r.glob(pat)
        if (p.parent / "preprocessed").is_dir()
    }
)
if len(_hits) != 1:
    _tree = sorted(str(p.relative_to("/kaggle/input")) for p in Path("/kaggle/input").glob("*/*"))
    raise FileNotFoundError(
        f"Expected exactly one dataset with preprocessed/ + splits.yaml under /kaggle/input, "
        f"found {[str(h) for h in _hits]}. Attach {DATA_SLUG}. Present: {_tree[:20]}"
    )
DATA = _hits[0]
PREP, SPLITS = DATA / "preprocessed", DATA / "splits.yaml"
n_cases = sum(1 for p in PREP.iterdir() if p.is_dir())
if n_cases == 0:
    raise FileNotFoundError(f"{PREP} exists but holds no case directories.")
print(n_cases, "preprocessed cases at", PREP)

## 5. Resume

`/kaggle/input` is read-only and `save_checkpoint` must write, so the previous session's
`last.pt` is copied into `/kaggle/working/checkpoints` first. `select_resume_checkpoint` then
finds it there on its own — the training call is identical for a fresh run and a resume.

With `CKPT_SLUG` set, anything other than exactly one `last.pt` raises. Missing it would not
error during training, it would just silently restart from epoch 0 — the expensive failure this
cell exists to prevent.

In [ ]:
CKPT_DIR = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
if CKPT_SLUG:
    # Searched across depths, not assumed at /kaggle/input/<slug>/ -- Kaggle
    # mounts sources one level deeper than documented (measured: the data
    # dataset landed at /kaggle/input/datasets/<owner>/<slug>/). Same reason
    # the data cell discovers rather than hardcodes.
    hits = sorted(set(Path("/kaggle/input").glob("**/last.pt")))
    if len(hits) != 1:
        mounted = sorted(str(q.relative_to("/kaggle/input")) for q in Path("/kaggle/input").glob("*/*"))
        raise FileNotFoundError(
            f"CKPT_SLUG={CKPT_SLUG!r}: want exactly one last.pt under /kaggle/input, found "
            f"{[str(h) for h in hits]}. Attach the checkpoint source, or set CKPT_SLUG=None "
            f"to start fresh. Mounted: {mounted[:20]}"
        )
    # /kaggle/input is read-only and save_checkpoint must write, so the file is
    # copied into the writable dir. find_resume_checkpoint then picks it up on
    # its own -- the training call is identical for a fresh run and a resume.
    shutil.copy2(hits[0], CKPT_DIR / "last.pt")
    import torch as _t
    _ck = _t.load(CKPT_DIR / "last.pt", weights_only=True, map_location="cpu")
    print(f"resuming from {hits[0]}")
    print(f"  epoch={_ck['epoch']} -> will start at {_ck['epoch']+1}, "
          f"best {_ck['best_metric_name']}={_ck['best_metric']:.4f}, wandb_run_id={_ck.get('wandb_run_id')}")
    del _ck

## 6. Compose config and train

`hydra.compose` with CLI-style overrides — the same strings `python scripts/train.py a=b` would
take. Calling `run_training` in-process (instead of shelling out) keeps the traceback in the
notebook and lets the cells above hand it already-validated paths.

The log's first line says `FRESH:` or `RESUME: ... from epoch N`. Check it. `max_hours: 11.0`
stops the run cleanly before Kaggle's 12-hour kill.

In [ ]:
import hydra

# sys.path for both src/ and scripts/ was set in the install cell, so this
# import cannot be the first place a missing package shows up.
from train import run_training

overrides = [f"data.root_dir={DATA}", f"data.preprocessing.out_dir={PREP}", f"data.splits.path={SPLITS}",
             f"training.checkpoint.dir={CKPT_DIR}"]
# experiment_name is passed ONLY when EXPERIMENT is set. An explicit
# experiment_name= override always beats a config group's own value, whatever
# the ordering -- Hydra applies group additions during composition and value
# overrides afterwards. So passing it unconditionally would silently rename an
# `+experiment=overfit2` run to EXPERIMENT, sending its checkpoints to
# outputs/<EXPERIMENT> and labelling its W&B run as that experiment. Set
# EXPERIMENT = None whenever OVERRIDES contains a `+experiment=` entry.
if EXPERIMENT is not None:
    overrides.append(f"experiment_name={EXPERIMENT}")
overrides += OVERRIDES

with hydra.initialize_config_dir(version_base="1.3", config_dir="/kaggle/working/repo/configs"):
    cfg = hydra.compose("config", overrides=overrides)
print("experiment_name =", cfg.experiment_name, "| epochs =", cfg.training.epochs)
metrics = run_training(cfg)

## 7. Verify the session output

Training already writes into `/kaggle/working/checkpoints`, which is the only path that survives
into the committed version's output — so there is nothing to copy, only to verify. Duplicating a
754 MB SwinUNETR checkpoint elsewhere under `/kaggle/working` would just eat the ~20 GB quota.

Attach this notebook version's output as input to the next session and set `CKPT_SLUG` to it.

In [ ]:
# Peak VRAM, reported rather than estimated. Three separate pre-run estimates
# of this model's memory were wrong (16 GB assumed capacity vs 14.56 actual,
# a 0.55 AMP factor that is really ~0.75+, and a per-patch figure read as a
# per-step one), so the run itself is now the source of truth. max_memory_*
# are process-lifetime peaks and survive the training call in the same kernel.
_total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(
    f"peak VRAM: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB allocated, "
    f"{torch.cuda.max_memory_reserved() / 2**30:.2f} GiB reserved, "
    f"of {_total:.2f} GiB total"
)

for name in ("last.pt", "best.pt"):
    p = CKPT_DIR / name
    if not p.is_file():
        raise FileNotFoundError(f"{p} missing — nothing to carry into the next session.")
    print(name, f"{p.stat().st_size / 2**20:.0f} MB  epoch={torch.load(p, weights_only=True)['epoch']}")
print(metrics)